# Hands-On Machine Learning — Chapter 8
## Dimensionality Reduction


### The Curse of Dimensionality

In high-dimensional spaces, data becomes sparse, distances lose meaning, and models often generalize poorly. This phenomenon is known as the **curse of dimensionality**. As the number of features increases, the volume of the feature space grows exponentially, and the data points become widely separated.

- A 1D interval of length 1 covers the range [0,1]. Dividing it into 10 segments gives 0.1 spacing.
- In 2D, 10×10 grid gives 100 cells. In 100D, you'd need 10^100 points to get the same density.

High dimensionality affects algorithms that rely on distances (e.g., kNN, clustering) or geometry (e.g., linear models, SVMs). Therefore, **reducing dimensions** helps improve generalization, visualization, and computational cost.

<p align="left"><img src="../fig/figure8.1.png" width="45%"></p>

The goal of dimensionality reduction is to represent data using fewer features while retaining most of its variance (information).

### Main Approaches to Dimensionality Reduction

There are two broad categories of methods:

1. **Projection Methods:** Project high-dimensional data onto a lower-dimensional subspace (e.g., PCA).
2. **Manifold Learning:** Discover a low-dimensional manifold embedded in the higher-dimensional space (e.g., LLE, t-SNE, Isomap).

<p align="left"><img src="../fig/figure8.2.png" width="45%"></p>

**Projection** assumes that data lies close to a linear subspace, whereas **manifold learning** assumes a nonlinear lower-dimensional structure.

### Principal Component Analysis (PCA)

**Principal Component Analysis (PCA)** is the most widely used linear dimensionality reduction technique. It identifies the directions (principal components) along which the data varies the most.

#### Theoretical Explanation
PCA computes the eigenvectors and eigenvalues of the data's covariance matrix:

$$\text{Cov}(X) = \frac{1}{m} X^T X$$

The eigenvectors correspond to the principal directions, and the eigenvalues represent the amount of variance explained by each component.

- The **first principal component (PC1)** captures the direction of maximum variance.
- The **second principal component (PC2)** is orthogonal to PC1 and captures the next highest variance.

<p align="left"><img src="../fig/figure8.3.png" width="45%"></p>

#### Dimensionality Reduction via PCA

PCA projects data onto the first *k* principal components to obtain a reduced representation:

$$ X_{reduced} = X W_k $$

where \(W_k\) contains the top *k* eigenvectors.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.datasets import load_iris

X, y = load_iris(return_X_y=True)
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X)

plt.figure(figsize=(6, 5))
plt.scatter(X_pca[:, 0], X_pca[:, 1], c=y, cmap='viridis', edgecolor='k')
plt.xlabel('Principal Component 1')
plt.ylabel('Principal Component 2')
plt.title('PCA Projection of the Iris Dataset')
plt.show()

### Explained Variance Ratio

The **explained variance ratio** indicates how much of the dataset's variance is captured by each principal component. It helps decide the number of components to retain.

<p align="left"><img src="../fig/figure8.4.png" width="45%"></p>

In practice, one often chooses the smallest number of components that explain, say, 95% of the variance.

In [ ]:
pca_full = PCA().fit(X)
cumulative_variance = np.cumsum(pca_full.explained_variance_ratio_)
plt.plot(cumulative_variance)
plt.xlabel('Number of components')
plt.ylabel('Cumulative explained variance')
plt.title('Explained Variance Ratio in PCA')
plt.grid(True)
plt.show()

### Reconstructing Data from Principal Components

Dimensionality reduction through PCA is lossy, but the original data can be approximately reconstructed:

$$ X_{approx} = X_{reduced} W_k^T $$

As the number of components increases, reconstruction error decreases. PCA balances between **compression** and **information retention**.

<p align="left"><img src="../fig/figure8.5.png" width="45%"></p>

### Incremental PCA

For very large datasets that don't fit into memory, **Incremental PCA (IPCA)** allows processing data in mini-batches. It updates the principal components incrementally, which is memory-efficient.

<p align="left"><img src="../fig/figure8.6.png" width="45%"></p>

In [ ]:
from sklearn.decomposition import IncrementalPCA

ipca = IncrementalPCA(n_components=2, batch_size=10)
X_ipca = ipca.fit_transform(X)
plt.scatter(X_ipca[:, 0], X_ipca[:, 1], c=y, cmap='plasma', edgecolor='k')
plt.xlabel('Component 1')
plt.ylabel('Component 2')
plt.title('Incremental PCA (batch processing)')
plt.show()

### Kernel PCA (kPCA)

**Kernel PCA** extends PCA to nonlinear structures by applying the **kernel trick**. Instead of explicitly computing new nonlinear features, it computes similarities using kernel functions (e.g., RBF, polynomial).

$$ K(x_i, x_j) = \phi(x_i)^T \phi(x_j) $$

Popular kernels:
- Polynomial: \(K(x, y) = (x^T y + 1)^d\)
- RBF (Gaussian): \(K(x, y) = \exp(-\gamma ||x - y||^2)\)

<p align="left"><img src="../fig/figure8.7.png" width="45%"></p>

In [ ]:
from sklearn.decomposition import KernelPCA
from sklearn.datasets import make_moons

X, y = make_moons(n_samples=100, noise=0.05, random_state=42)
kpca = KernelPCA(n_components=2, kernel='rbf', gamma=15)
X_kpca = kpca.fit_transform(X)

plt.figure(figsize=(6, 5))
plt.scatter(X_kpca[:, 0], X_kpca[:, 1], c=y, cmap='coolwarm', edgecolor='k')
plt.title('Kernel PCA (RBF kernel) Projection')
plt.show()

### Locally Linear Embedding (LLE)

LLE is a **manifold learning** technique that assumes each data point can be linearly reconstructed from its nearest neighbors. It finds a lower-dimensional embedding that preserves these local relationships.

<p align="left"><img src="../fig/figure8.8.png" width="45%"></p>

Mathematically, LLE minimizes the reconstruction error:

$$ \sum_i ||x_i - \sum_j w_{ij} x_j||^2 $$

subject to weights \(w_{ij}\) summing to 1 for each data point.

In [ ]:
from sklearn.manifold import LocallyLinearEmbedding

lle = LocallyLinearEmbedding(n_neighbors=10, n_components=2)
X_lle = lle.fit_transform(X)
plt.scatter(X_lle[:, 0], X_lle[:, 1], c=y, cmap='Spectral', edgecolor='k')
plt.title('Locally Linear Embedding (LLE)')
plt.show()

### Choosing the Right Dimensionality Reduction Method

When selecting a technique:
- **PCA:** good default, fast, linear relationships.
- **Incremental PCA:** for large datasets.
- **Kernel PCA:** for nonlinear manifolds.
- **LLE:** for manifold structure with local relationships.

<p align="left"><img src="../fig/figure8.14.png" width="45%"></p>

In practice, try PCA first; if reconstruction or visualization fails, move to nonlinear methods like Kernel PCA or LLE. Modern alternatives include **t-SNE** and **UMAP** for high-quality visualization.